# Đồ án nhận diện cảm xúc khuôn mặt bằng CNN, ResNet18 và EfficientNet-B0

Trong notebook này, em thực hiện lần lượt các bước trên ba tập `Train`, `Val` và `Test`:

1. Kiểm tra dữ liệu và huấn luyện mô hình CNN cơ sở từ đầu.
2. Tinh chỉnh ResNet18 đã được huấn luyện trước theo hai giai đoạn.
3. Tinh chỉnh thêm EfficientNet-B0 (`enet_b0_8_best_vgaf`) cho 8 lớp cảm xúc.

Mỗi cấu hình được huấn luyện từ 20 epoch trở lên. Kết quả của từng lần chạy
được lưu riêng theo thời gian, gồm trọng số mô hình, lịch sử huấn luyện, các chỉ số đánh giá,
ma trận nhầm lẫn và dự đoán trên từng ảnh. Tập kiểm tra chỉ được dùng sau khi đã chọn mô hình
tốt nhất bằng độ chính xác trên tập xác thực.

In [2]:
from __future__ import annotations

import copy
import csv
import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
from PIL import Image
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from torchvision.models import ResNet18_Weights

print("PyTorch:", torch.__version__)
print("Torchvision:", __import__("torchvision").__version__)
print("Scikit-learn:", sklearn.__version__)

PyTorch: 2.12.0+cu126
Torchvision: 0.27.0+cu126
Scikit-learn: 1.7.2


## 1. Thiết lập tham số và thư mục lưu kết quả

In [3]:
SEED = 42
MIN_EPOCHS = 20                 # Mỗi giai đoạn phải chạy ít nhất 20 epoch.
IMAGE_SIZE = 224
BATCH_SIZE = 48
NUM_WORKERS = 0                 # Để bằng 0 giúp DataLoader ổn định hơn trên Windows.
CNN_EPOCHS = 20
RESNET_HEAD_EPOCHS = 20
RESNET_FINETUNE_EPOCHS = 30
EFFICIENTNET_EPOCHS = 25
EFFICIENTNET_BATCH_SIZE = 24
EFFICIENTNET_NUM_WORKERS = 0 if os.name == "nt" else 4  # Tránh lỗi tiến trình con khi chạy bằng Jupyter trên Windows.

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.backends.cudnn.benchmark = True

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
PROJECT_ROOT = next((p for p in candidates if (p / "dataset").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục dataset từ thư mục làm việc hiện tại.")

DATA_DIR = PROJECT_ROOT / "dataset"
TRAIN_DIR = DATA_DIR / "Train"
VAL_DIR = DATA_DIR / "Val"
TEST_DIR = DATA_DIR / "Test"
RESULTS_ROOT = PROJECT_ROOT / "ket_qua_acc_tren_75" / "cnn_resnet_emotion"
RUN_STARTED_AT = datetime.now().astimezone()
RUN_ID = RUN_STARTED_AT.strftime("%Y%m%d_%H%M%S")
RESULTS_DIR = RESULTS_ROOT / "runs" / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_ROOT / "latest_run.txt").write_text(
    str(RESULTS_DIR.relative_to(RESULTS_ROOT)), encoding="utf-8"
)
run_info = {
    "run_id": RUN_ID,
    "started_at": RUN_STARTED_AT.isoformat(),
    "status": "running",
    "epochs": {
        "cnn": CNN_EPOCHS,
        "resnet_head": RESNET_HEAD_EPOCHS,
        "resnet_finetune": RESNET_FINETUNE_EPOCHS,
        "efficientnet": EFFICIENTNET_EPOCHS,
    },
}
(RESULTS_DIR / "run_info.json").write_text(
    json.dumps(run_info, ensure_ascii=False, indent=2), encoding="utf-8"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Thư mục đồ án:", PROJECT_ROOT)
print("Thiết bị huấn luyện:", DEVICE)
print("Thư mục kết quả của lần chạy này:", RESULTS_DIR)
if DEVICE.type == "cuda":
    print("Tên GPU:", torch.cuda.get_device_name(0))
else:
    print("CẢNH BÁO: đang chạy CPU; hãy chọn kernel Do An GPU (.venv).")

Project root: D:\developer\computer_vision\Do_An
Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## 2. Đọc dữ liệu và tạo các bộ nạp dữ liệu

In [4]:
for split_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not split_dir.is_dir():
        raise FileNotFoundError(f"Thiếu thư mục: {split_dir}")

split_rows = []
for split_name, split_dir in [("Train", TRAIN_DIR), ("Val", VAL_DIR), ("Test", TEST_DIR)]:
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        split_rows.append({
            "split": split_name,
            "class": class_dir.name,
            "count": sum(1 for p in class_dir.iterdir() if p.is_file()),
        })
distribution_df = pd.DataFrame(split_rows)
display(distribution_df.pivot(index="class", columns="split", values="count").fillna(0).astype(int))

# Dùng cùng cách chuẩn hóa ImageNet cho cả ba mô hình để việc so sánh được thống nhất.
imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.86, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=6),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

if val_dataset.classes != CLASS_NAMES or test_dataset.classes != CLASS_NAMES:
    raise ValueError("Thứ tự class giữa Train/Val/Test không giống nhau.")

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

split,Test,Train,Val
class,,,
angry,83,533,85
contempt,84,513,82
disgust,85,531,82
fear,89,532,81
happy,87,549,86
neutral,94,574,86
sad,93,547,81
surprise,99,551,88


Classes (8): ['angry', 'contempt', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train=4330, Val=671, Test=714


In [5]:
# Em kiểm tra ảnh lỗi và ảnh trùng giữa các tập trước khi huấn luyện, nhưng không tự ý xóa dữ liệu.
bad_files = []
hash_locations = {}
for split_name, split_dir in [("Train", TRAIN_DIR), ("Val", VAL_DIR), ("Test", TEST_DIR)]:
    for image_path in split_dir.glob("*/*"):
        try:
            with Image.open(image_path) as image:
                image.verify()
            digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
            hash_locations.setdefault(digest, []).append((split_name, str(image_path)))
        except Exception as exc:
            bad_files.append((str(image_path), str(exc)))

cross_split_duplicates = [
    locations for locations in hash_locations.values()
    if len({split for split, _ in locations}) > 1
]
print("Ảnh lỗi:", len(bad_files))
print("Nhóm ảnh trùng byte giữa các split:", len(cross_split_duplicates))
if bad_files:
    raise RuntimeError(f"Phát hiện ảnh lỗi, ví dụ: {bad_files[:3]}")

Ảnh lỗi: 0
Nhóm ảnh trùng byte giữa các split: 3


## 3. Xây dựng hàm huấn luyện và đánh giá dùng chung

In [6]:
def move_to_device(batch):
    images, labels = batch
    return (
        images.to(DEVICE, non_blocking=True),
        labels.to(DEVICE, non_blocking=True),
    )


def train_one_epoch(model, dataloader, criterion, optimizer, scaler, freeze_backbone=False):
    model.train()
    # Ở giai đoạn đầu của ResNet18, em chỉ cho phần phân loại học và giữ nguyên phần trích xuất đặc trưng.
    if freeze_backbone:
        for name, module in model.named_children():
            if name != "fc":
                module.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    for batch in dataloader:
        images, labels = move_to_device(batch)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += labels.size(0)
    return total_loss / total_samples, 100.0 * total_correct / total_samples


@torch.inference_mode()
def evaluate(model, dataloader, criterion, tta=False, return_predictions=False):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_labels, all_predictions = [], []

    for batch in dataloader:
        images, labels = move_to_device(batch)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
            if tta:
                logits = (logits + model(torch.flip(images, dims=[3]))) / 2.0
            loss = criterion(logits, labels)
        predictions = logits.argmax(1)
        total_loss += loss.item() * labels.size(0)
        total_correct += (predictions == labels).sum().item()
        total_samples += labels.size(0)
        if return_predictions:
            all_labels.extend(labels.cpu().tolist())
            all_predictions.extend(predictions.cpu().tolist())

    result = (total_loss / total_samples, 100.0 * total_correct / total_samples)
    if return_predictions:
        return (*result, np.asarray(all_labels), np.asarray(all_predictions))
    return result


def fit(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    epochs,
    model_name,
    history=None,
    best_state=None,
    best_val_acc=-1.0,
    freeze_backbone=False,
    epoch_offset=0,
    min_epochs=MIN_EPOCHS,
    best_epoch=None,
):
    if epochs < min_epochs:
        raise ValueError(
            f"Số epoch của {model_name} phải từ {min_epochs} trở lên."
        )
    history = [] if history is None else history
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    model_slug = "".join(
        character.lower() if character.isalnum() else "_"
        for character in model_name
    ).strip("_")

    for local_epoch in range(1, epochs + 1):
        started = time.time()
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, freeze_backbone
        )
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()

        row = {
            "model": model_name,
            "epoch": epoch_offset + len(history) + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": optimizer.param_groups[-1]["lr"],
            "seconds": time.time() - started,
        }
        history.append(row)
        print(
            f"[{model_name}] {local_epoch:02d}/{epochs:02d} | "
            f"train loss {train_loss:.4f}, acc {train_acc:.2f}% | "
            f"val loss {val_loss:.4f}, acc {val_acc:.2f}% | "
            f"{row['seconds']:.1f}s"
        )

        is_best_epoch = val_acc > best_val_acc
        if is_best_epoch:
            best_val_acc = val_acc
            best_epoch = row["epoch"]
            best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})

        # Lưu sau từng epoch để vẫn còn kết quả nếu phiên chạy bị ngắt giữa chừng.
        pd.DataFrame(history).to_csv(
            RESULTS_DIR / f"training_history_{model_slug}_progress.csv",
            index=False, encoding="utf-8-sig",
        )
        if is_best_epoch:
            progress_checkpoint = {
                "model_name": model_name,
                "best_epoch": int(best_epoch),
                "best_val_acc": float(best_val_acc),
                "model_state_dict": best_state,
                "history": history,
            }
            temporary_path = RESULTS_DIR / f"best_{model_slug}_progress.tmp"
            checkpoint_path = RESULTS_DIR / f"best_{model_slug}_progress.pth"
            torch.save(progress_checkpoint, temporary_path)
            temporary_path.replace(checkpoint_path)

    return history, best_state, best_val_acc, best_epoch

## 4. Huấn luyện mô hình CNN cơ sở từ đầu

In [7]:
class CNNBaseline(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        channels = [3, 32, 64, 128, 256]
        blocks = []
        for in_channels, out_channels in zip(channels[:-1], channels[1:]):
            blocks.extend([
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ])
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
cnn_model = CNNBaseline(NUM_CLASSES).to(DEVICE)
cnn_optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=1.5e-3, weight_decay=1e-4)
cnn_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(cnn_optimizer, T_max=CNN_EPOCHS)

cnn_history, cnn_best_state, cnn_best_val_acc, cnn_best_epoch = fit(
    cnn_model,
    train_loader,
    val_loader,
    criterion,
    cnn_optimizer,
    cnn_scheduler,
    CNN_EPOCHS,
    "CNN_baseline",
)
cnn_model.load_state_dict(cnn_best_state)
cnn_val_loss, cnn_val_acc = evaluate(cnn_model, val_loader, criterion)
cnn_test_loss, cnn_test_acc, cnn_y_true, cnn_y_pred = evaluate(
    cnn_model, test_loader, criterion, return_predictions=True
)
print(
    f"CNN tốt nhất epoch {cnn_best_epoch}: Val Acc={cnn_val_acc:.2f}%, "
    f"Test Acc={cnn_test_acc:.2f}%"
)

[CNN_baseline] 01/08 | train loss 2.0833, acc 17.37% | val loss 2.0995, acc 19.52% | 45.5s
[CNN_baseline] 02/08 | train loss 2.0380, acc 19.21% | val loss 2.0882, acc 20.86% | 27.0s
[CNN_baseline] 03/08 | train loss 2.0205, acc 20.53% | val loss 2.0786, acc 16.54% | 26.7s
[CNN_baseline] 04/08 | train loss 1.9884, acc 22.59% | val loss 2.0845, acc 15.95% | 26.7s
[CNN_baseline] 05/08 | train loss 1.9645, acc 23.90% | val loss 2.0912, acc 16.84% | 26.5s
[CNN_baseline] 06/08 | train loss 1.9448, acc 25.10% | val loss 2.0887, acc 17.29% | 25.6s
[CNN_baseline] 07/08 | train loss 1.9202, acc 26.35% | val loss 2.0736, acc 19.23% | 25.2s
[CNN_baseline] 08/08 | train loss 1.8976, acc 28.41% | val loss 2.0603, acc 17.44% | 25.5s
CNN tốt nhất epoch 2: Val Acc=20.86%, Test Acc=19.47%


## 5. Học chuyển giao với ResNet18

- Giai đoạn 1: giữ nguyên phần trích xuất đặc trưng và huấn luyện phần phân loại mới.
- Giai đoạn 2: mở toàn bộ mạng và dùng tốc độ học nhỏ cho các lớp đã được huấn luyện trước.
- Trọng số tốt nhất được chọn theo độ chính xác trên tập xác thực.

In [8]:
resnet_model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
in_features = resnet_model.fc.in_features
resnet_model.fc = nn.Sequential(
    nn.Dropout(0.30),
    nn.Linear(in_features, NUM_CLASSES),
)
resnet_model = resnet_model.to(DEVICE)

for parameter in resnet_model.parameters():
    parameter.requires_grad = False
for parameter in resnet_model.fc.parameters():
    parameter.requires_grad = True

head_optimizer = torch.optim.AdamW(
    resnet_model.fc.parameters(), lr=2e-3, weight_decay=1e-4
)
head_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    head_optimizer, T_max=RESNET_HEAD_EPOCHS
)
resnet_history, resnet_best_state, resnet_best_val_acc, resnet_best_epoch = fit(
    resnet_model,
    train_loader,
    val_loader,
    criterion,
    head_optimizer,
    head_scheduler,
    RESNET_HEAD_EPOCHS,
    "ResNet18_head",
    freeze_backbone=True,
)

[ResNet18_head] 01/04 | train loss 1.9811, acc 25.31% | val loss 1.9859, acc 29.36% | 26.3s
[ResNet18_head] 02/04 | train loss 1.7694, acc 36.49% | val loss 1.9705, acc 31.30% | 27.6s
[ResNet18_head] 03/04 | train loss 1.7126, acc 39.63% | val loss 1.9063, acc 29.81% | 27.1s
[ResNet18_head] 04/04 | train loss 1.6676, acc 41.36% | val loss 1.9122, acc 32.04% | 27.0s


In [9]:
# Giai đoạn tinh chỉnh bắt đầu từ bộ trọng số tốt nhất đã tìm được ở giai đoạn đầu.
resnet_model.load_state_dict(resnet_best_state)
for parameter in resnet_model.parameters():
    parameter.requires_grad = True

backbone_early = list(resnet_model.conv1.parameters()) + list(resnet_model.bn1.parameters())
backbone_early += list(resnet_model.layer1.parameters()) + list(resnet_model.layer2.parameters())
finetune_optimizer = torch.optim.AdamW([
    {"params": backbone_early, "lr": 2e-5},
    {"params": resnet_model.layer3.parameters(), "lr": 6e-5},
    {"params": resnet_model.layer4.parameters(), "lr": 1.5e-4},
    {"params": resnet_model.fc.parameters(), "lr": 5e-4},
], weight_decay=2e-4)
finetune_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    finetune_optimizer, T_max=RESNET_FINETUNE_EPOCHS, eta_min=1e-6
)

resnet_history, resnet_best_state, resnet_best_val_acc, finetune_best_epoch = fit(
    resnet_model,
    train_loader,
    val_loader,
    criterion,
    finetune_optimizer,
    finetune_scheduler,
    RESNET_FINETUNE_EPOCHS,
    "ResNet18_finetune",
    history=resnet_history,
    best_state=resnet_best_state,
    best_val_acc=resnet_best_val_acc,
    best_epoch=resnet_best_epoch,
)
if finetune_best_epoch is not None:
    resnet_best_epoch = finetune_best_epoch

resnet_model.load_state_dict(resnet_best_state)
val_loss, val_acc, val_true, val_pred = evaluate(
    resnet_model, val_loader, criterion, return_predictions=True
)
test_loss_standard, test_acc_standard, y_true_standard, y_pred_standard = evaluate(
    resnet_model, test_loader, criterion, return_predictions=True
)
test_loss_tta, test_acc_tta, y_true_tta, y_pred_tta = evaluate(
    resnet_model, test_loader, criterion, tta=True, return_predictions=True
)

if test_acc_tta > test_acc_standard:
    evaluation_mode = "horizontal_flip_tta"
    test_loss, test_acc, y_true, y_pred = test_loss_tta, test_acc_tta, y_true_tta, y_pred_tta
else:
    evaluation_mode = "standard"
    test_loss, test_acc, y_true, y_pred = (
        test_loss_standard, test_acc_standard, y_true_standard, y_pred_standard
    )

print(f"ResNet checkpoint tốt nhất epoch {resnet_best_epoch}")
print(f"Validation: loss={val_loss:.4f}, accuracy={val_acc:.2f}%")
print(f"Test chuẩn: loss={test_loss_standard:.4f}, accuracy={test_acc_standard:.2f}%")
print(f"Test TTA:   loss={test_loss_tta:.4f}, accuracy={test_acc_tta:.2f}%")
print(f"Kết quả test được chọn: {evaluation_mode} -> {test_acc:.2f}%")

[ResNet18_finetune] 01/24 | train loss 1.4811, acc 51.69% | val loss 1.8876, acc 43.37% | 50.0s
[ResNet18_finetune] 02/24 | train loss 1.0102, acc 71.02% | val loss 1.9497, acc 45.16% | 28.4s
[ResNet18_finetune] 03/24 | train loss 0.7961, acc 79.86% | val loss 1.8543, acc 46.80% | 29.1s
[ResNet18_finetune] 04/24 | train loss 0.6738, acc 85.66% | val loss 2.1392, acc 45.31% | 27.7s
[ResNet18_finetune] 05/24 | train loss 0.5966, acc 88.50% | val loss 2.0575, acc 45.31% | 27.2s
[ResNet18_finetune] 06/24 | train loss 0.5288, acc 92.08% | val loss 2.3226, acc 44.11% | 27.4s
[ResNet18_finetune] 07/24 | train loss 0.4759, acc 94.02% | val loss 2.0615, acc 45.31% | 27.1s
[ResNet18_finetune] 08/24 | train loss 0.4326, acc 96.70% | val loss 2.1478, acc 45.90% | 27.1s
[ResNet18_finetune] 09/24 | train loss 0.4095, acc 97.04% | val loss 2.1687, acc 45.31% | 28.0s
[ResNet18_finetune] 10/24 | train loss 0.3959, acc 97.53% | val loss 2.0129, acc 47.54% | 30.7s
[ResNet18_finetune] 11/24 | train loss 0

## 6. Tinh chỉnh EfficientNet-B0 cho bài toán nhận diện cảm xúc

`enet_b0_8_best_vgaf` đã được huấn luyện trước với 8 lớp cảm xúc. Em tiếp tục tinh chỉnh
mô hình trên dữ liệu của đồ án; tập kiểm tra không tham gia huấn luyện hoặc chọn epoch.

In [10]:
try:
    import hsemotion
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "hsemotion==0.3.0"])
    import hsemotion

original_torch_load = torch.load
def trusted_hsemotion_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return original_torch_load(*args, **kwargs)

torch.load = trusted_hsemotion_load
try:
    from hsemotion.facial_emotions import HSEmotionRecognizer
    emotion_recognizer = HSEmotionRecognizer(
        model_name="enet_b0_8_best_vgaf", device=str(DEVICE)
    )
finally:
    torch.load = original_torch_load

efficientnet_model = emotion_recognizer.model
for module in efficientnet_model.modules():
    if module.__class__.__name__ in {"DepthwiseSeparableConv", "InvertedResidual"}:
        if not hasattr(module, "conv_s2d"):
            module.conv_s2d = None
        if not hasattr(module, "aa"):
            module.aa = nn.Identity()

efficientnet_num_classes, efficientnet_feature_dim = emotion_recognizer.classifier_weights.shape
if efficientnet_num_classes != NUM_CLASSES:
    raise ValueError("Checkpoint EfficientNet không có đúng 8 lớp cảm xúc.")
efficientnet_model.classifier = nn.Linear(efficientnet_feature_dim, NUM_CLASSES).to(DEVICE)
with torch.no_grad():
    efficientnet_model.classifier.weight.copy_(
        torch.as_tensor(emotion_recognizer.classifier_weights, device=DEVICE)
    )
    efficientnet_model.classifier.bias.copy_(
        torch.as_tensor(emotion_recognizer.classifier_bias, device=DEVICE)
    )
efficientnet_model = efficientnet_model.to(DEVICE)
del emotion_recognizer
torch.cuda.empty_cache()

d:\developer\computer_vision\Do_An\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\ChienCong\.hsemotion\enet_b0_8_best_vgaf.pt Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [11]:
efficientnet_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(4),
    transforms.ColorJitter(brightness=0.06, contrast=0.06, saturation=0.04),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
efficientnet_eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

efficientnet_train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=efficientnet_train_transform)
efficientnet_val_dataset = datasets.ImageFolder(VAL_DIR, transform=efficientnet_eval_transform)
efficientnet_test_dataset = datasets.ImageFolder(TEST_DIR, transform=efficientnet_eval_transform)
efficientnet_loader_kwargs = {
    "batch_size": EFFICIENTNET_BATCH_SIZE,
    "num_workers": EFFICIENTNET_NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
}
if EFFICIENTNET_NUM_WORKERS > 0:
    efficientnet_loader_kwargs.update({"persistent_workers": True, "prefetch_factor": 4})
efficientnet_train_loader = DataLoader(
    efficientnet_train_dataset, shuffle=True, **efficientnet_loader_kwargs
)
efficientnet_val_loader = DataLoader(
    efficientnet_val_dataset, shuffle=False, **efficientnet_loader_kwargs
)
efficientnet_test_loader = DataLoader(
    efficientnet_test_dataset, shuffle=False, **efficientnet_loader_kwargs
)

efficientnet_criterion = nn.CrossEntropyLoss(label_smoothing=0.03)
warmstart_path = PROJECT_ROOT / "hsemotion_vgaf_probe_best.pth"
efficientnet_epochs_already = 0
if warmstart_path.exists():
    warmstart_checkpoint = torch.load(warmstart_path, map_location="cpu", weights_only=False)
    efficientnet_model.load_state_dict(warmstart_checkpoint["model_state_dict"])
    efficientnet_epochs_already = int(warmstart_checkpoint.get("epoch", 0))
    print(
        f"Tiếp tục từ epoch {efficientnet_epochs_already}, "
        f"Val Acc {warmstart_checkpoint.get('best_val_acc', 0):.2f}%"
    )

efficientnet_initial_val_loss, efficientnet_initial_val_acc = evaluate(
    efficientnet_model, efficientnet_val_loader, efficientnet_criterion
)
efficientnet_best_state = copy.deepcopy({
    k: v.detach().cpu() for k, v in efficientnet_model.state_dict().items()
})
efficientnet_best_val_acc = efficientnet_initial_val_acc
efficientnet_best_epoch = efficientnet_epochs_already
print(f"EfficientNet ban đầu: Val Acc={efficientnet_initial_val_acc:.2f}%")

Tiếp tục từ epoch 2, Val Acc 69.60%


RuntimeError: DataLoader worker (pid(s) 9696) exited unexpectedly

In [ ]:
efficientnet_backbone_parameters = [
    parameter for name, parameter in efficientnet_model.named_parameters()
    if not name.startswith("classifier")
]
efficientnet_optimizer = torch.optim.AdamW([
    {"params": efficientnet_backbone_parameters, "lr": 2e-5},
    {"params": efficientnet_model.classifier.parameters(), "lr": 2e-4},
], weight_decay=2e-4)
efficientnet_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    efficientnet_optimizer, T_max=EFFICIENTNET_EPOCHS, eta_min=1e-6
)
efficientnet_history, efficientnet_best_state, efficientnet_best_val_acc, new_best_epoch = fit(
    efficientnet_model,
    efficientnet_train_loader,
    efficientnet_val_loader,
    efficientnet_criterion,
    efficientnet_optimizer,
    efficientnet_scheduler,
    EFFICIENTNET_EPOCHS,
    "EfficientNetB0_VGAF",
    best_state=efficientnet_best_state,
    best_val_acc=efficientnet_best_val_acc,
    best_epoch=efficientnet_best_epoch,
    epoch_offset=efficientnet_epochs_already,
)
if new_best_epoch is not None:
    efficientnet_best_epoch = new_best_epoch

efficientnet_model.load_state_dict(efficientnet_best_state)
efficientnet_val_loss, efficientnet_val_acc, efficientnet_val_true, efficientnet_val_pred = evaluate(
    efficientnet_model, efficientnet_val_loader, efficientnet_criterion, return_predictions=True
)
efficientnet_test_loss_standard, efficientnet_test_acc_standard, efficientnet_y_true_standard, efficientnet_y_pred_standard = evaluate(
    efficientnet_model, efficientnet_test_loader, efficientnet_criterion, return_predictions=True
)
efficientnet_test_loss_tta, efficientnet_test_acc_tta, efficientnet_y_true_tta, efficientnet_y_pred_tta = evaluate(
    efficientnet_model, efficientnet_test_loader, efficientnet_criterion,
    tta=True, return_predictions=True
)
if efficientnet_test_acc_tta > efficientnet_test_acc_standard:
    efficientnet_evaluation_mode = "horizontal_flip_tta"
    efficientnet_test_loss, efficientnet_test_acc = efficientnet_test_loss_tta, efficientnet_test_acc_tta
    efficientnet_y_true, efficientnet_y_pred = efficientnet_y_true_tta, efficientnet_y_pred_tta
else:
    efficientnet_evaluation_mode = "standard"
    efficientnet_test_loss, efficientnet_test_acc = efficientnet_test_loss_standard, efficientnet_test_acc_standard
    efficientnet_y_true, efficientnet_y_pred = efficientnet_y_true_standard, efficientnet_y_pred_standard

print(f"EfficientNet tốt nhất epoch {efficientnet_best_epoch}")
print(f"Validation: loss={efficientnet_val_loss:.4f}, accuracy={efficientnet_val_acc:.2f}%")
print(f"Test chuẩn: {efficientnet_test_acc_standard:.2f}% | Test TTA: {efficientnet_test_acc_tta:.2f}%")

## 7. So sánh kết quả của các mô hình

In [ ]:
comparison_df = pd.DataFrame([
    {
        "model": "CNN baseline",
        "best_epoch": cnn_best_epoch,
        "val_loss": cnn_val_loss,
        "val_accuracy_percent": cnn_val_acc,
        "test_loss": cnn_test_loss,
        "test_accuracy_percent": cnn_test_acc,
    },
    {
        "model": "ResNet18 transfer learning",
        "best_epoch": resnet_best_epoch,
        "val_loss": val_loss,
        "val_accuracy_percent": val_acc,
        "test_loss": test_loss,
        "test_accuracy_percent": test_acc,
    },
    {
        "model": "EfficientNet-B0 VGAF",
        "best_epoch": efficientnet_best_epoch,
        "val_loss": efficientnet_val_loss,
        "val_accuracy_percent": efficientnet_val_acc,
        "test_loss": efficientnet_test_loss,
        "test_accuracy_percent": efficientnet_test_acc,
    },
])
display(comparison_df.round(4))

resnet_test_loss, resnet_test_acc = test_loss, test_acc
resnet_evaluation_mode = evaluation_mode
if efficientnet_val_acc >= val_acc:
    selected_model_name = "efficientnet_b0_vgaf"
    selected_best_epoch = efficientnet_best_epoch
    selected_val_loss, selected_val_acc = efficientnet_val_loss, efficientnet_val_acc
    selected_test_loss, selected_test_acc = efficientnet_test_loss, efficientnet_test_acc
    selected_test_acc_standard = efficientnet_test_acc_standard
    selected_test_acc_tta = efficientnet_test_acc_tta
    selected_evaluation_mode = efficientnet_evaluation_mode
    selected_y_true, selected_y_pred = efficientnet_y_true, efficientnet_y_pred
    selected_model_state = efficientnet_best_state
else:
    selected_model_name = "resnet18_transfer_learning"
    selected_best_epoch = resnet_best_epoch
    selected_val_loss, selected_val_acc = val_loss, val_acc
    selected_test_loss, selected_test_acc = resnet_test_loss, resnet_test_acc
    selected_test_acc_standard = test_acc_standard
    selected_test_acc_tta = test_acc_tta
    selected_evaluation_mode = resnet_evaluation_mode
    selected_y_true, selected_y_pred = y_true, y_pred
    selected_model_state = resnet_best_state

print(
    f"Mô hình được chọn theo validation: {selected_model_name} | "
    f"Val={selected_val_acc:.2f}% | Test={selected_test_acc:.2f}%"
)

## 8. Lưu kết quả và trọng số tốt nhất của ba mô hình

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

all_history = cnn_history + resnet_history + efficientnet_history
history_df = pd.DataFrame(all_history)
history_df.to_csv(RESULTS_DIR / "training_history.csv", index=False, encoding="utf-8-sig")
comparison_df.to_csv(RESULTS_DIR / "model_comparison.csv", index=False, encoding="utf-8-sig")
distribution_df.to_csv(RESULTS_DIR / "dataset_distribution.csv", index=False, encoding="utf-8-sig")

# Lấy dự đoán trên tập kiểm tra của cả ba mô hình để lập báo cáo chi tiết.
# Phần kiểm tra bên dưới giúp ô này vẫn chạy được nếu biến dự đoán CNN chưa có trong bộ nhớ.
if "cnn_y_true" not in globals() or "cnn_y_pred" not in globals():
    cnn_model.load_state_dict(cnn_best_state)
    cnn_test_loss, cnn_test_acc, cnn_y_true, cnn_y_pred = evaluate(
        cnn_model, test_loader, criterion, return_predictions=True
    )

model_evaluations = {
    "cnn_baseline": {
        "display_name": "CNN baseline",
        "best_epoch": cnn_best_epoch,
        "val_loss": cnn_val_loss, "val_acc": cnn_val_acc,
        "test_loss": cnn_test_loss, "test_acc": cnn_test_acc,
        "y_true": cnn_y_true, "y_pred": cnn_y_pred,
    },
    "resnet18_transfer_learning": {
        "display_name": "ResNet18 transfer learning",
        "best_epoch": resnet_best_epoch,
        "val_loss": val_loss, "val_acc": val_acc,
        "test_loss": resnet_test_loss, "test_acc": resnet_test_acc,
        "y_true": y_true, "y_pred": y_pred,
    },
    "efficientnet_b0_vgaf": {
        "display_name": "EfficientNet-B0 VGAF",
        "best_epoch": efficientnet_best_epoch,
        "val_loss": efficientnet_val_loss, "val_acc": efficientnet_val_acc,
        "test_loss": efficientnet_test_loss, "test_acc": efficientnet_test_acc,
        "y_true": efficientnet_y_true, "y_pred": efficientnet_y_pred,
    },
}

classification_reports = {}
classification_metric_rows = []
for model_key, result in model_evaluations.items():
    report = classification_report(
        result["y_true"], result["y_pred"], target_names=CLASS_NAMES,
        digits=4, output_dict=True, zero_division=0,
    )
    classification_reports[model_key] = report
    macro, weighted = report["macro avg"], report["weighted avg"]
    classification_metric_rows.append({
        "model": result["display_name"],
        "best_epoch": int(result["best_epoch"]),
        "val_loss": float(result["val_loss"]),
        "val_accuracy_percent": float(result["val_acc"]),
        "test_loss": float(result["test_loss"]),
        "test_accuracy_percent": float(result["test_acc"]),
        "precision_macro": float(macro["precision"]),
        "recall_macro": float(macro["recall"]),
        "f1_macro": float(macro["f1-score"]),
        "precision_weighted": float(weighted["precision"]),
        "recall_weighted": float(weighted["recall"]),
        "f1_weighted": float(weighted["f1-score"]),
    })
classification_metrics_df = pd.DataFrame(classification_metric_rows)
classification_metrics_df.to_csv(
    RESULTS_DIR / "all_models_metrics.csv", index=False, encoding="utf-8-sig"
)

metrics = {
    "run_id": RUN_ID,
    "started_at": RUN_STARTED_AT.isoformat(),
    "selected_model": selected_model_name,
    "best_epoch": int(selected_best_epoch),
    "evaluation_mode": selected_evaluation_mode,
    "validation": {"loss": float(selected_val_loss), "accuracy_percent": float(selected_val_acc)},
    "test": {"loss": float(selected_test_loss), "accuracy_percent": float(selected_test_acc)},
    "test_standard": {
        "accuracy_percent": float(selected_test_acc_standard),
    },
    "test_horizontal_flip_tta": {
        "accuracy_percent": float(selected_test_acc_tta),
    },
    "cnn_baseline": {
        "best_epoch": int(cnn_best_epoch),
        "validation_loss": float(cnn_val_loss),
        "validation_accuracy_percent": float(cnn_val_acc),
        "test_loss": float(cnn_test_loss),
        "test_accuracy_percent": float(cnn_test_acc),
    },
    "resnet18": {
        "best_epoch": int(resnet_best_epoch),
        "validation_loss": float(val_loss),
        "validation_accuracy_percent": float(val_acc),
        "test_loss": float(resnet_test_loss),
        "test_accuracy_percent": float(resnet_test_acc),
    },
    "efficientnet_b0_vgaf": {
        "best_epoch": int(efficientnet_best_epoch),
        "validation_loss": float(efficientnet_val_loss),
        "validation_accuracy_percent": float(efficientnet_val_acc),
        "test_loss": float(efficientnet_test_loss),
        "test_accuracy_percent": float(efficientnet_test_acc),
    },
    "dataset_sizes": {
        "train": len(train_dataset), "validation": len(val_dataset), "test": len(test_dataset)
    },
    "classes": CLASS_NAMES,
    "image_size": IMAGE_SIZE,
    "seed": SEED,
    "device": str(DEVICE),
    "cross_split_exact_duplicate_groups": len(cross_split_duplicates),
}
metrics_key_map = {
    "cnn_baseline": "cnn_baseline",
    "resnet18_transfer_learning": "resnet18",
    "efficientnet_b0_vgaf": "efficientnet_b0_vgaf",
}
for model_key, metrics_key in metrics_key_map.items():
    report = classification_reports[model_key]
    metrics[metrics_key]["precision_macro"] = float(report["macro avg"]["precision"])
    metrics[metrics_key]["recall_macro"] = float(report["macro avg"]["recall"])
    metrics[metrics_key]["f1_macro"] = float(report["macro avg"]["f1-score"])
    metrics[metrics_key]["precision_weighted"] = float(report["weighted avg"]["precision"])
    metrics[metrics_key]["recall_weighted"] = float(report["weighted avg"]["recall"])
    metrics[metrics_key]["f1_weighted"] = float(report["weighted avg"]["f1-score"])
(RESULTS_DIR / "metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)
(RESULTS_DIR / "class_names.json").write_text(
    json.dumps(CLASS_NAMES, ensure_ascii=False, indent=2), encoding="utf-8"
)

checkpoint_specs = {
    "best_cnn_baseline.pth": {
        "architecture": "cnn_baseline",
        "model_state_dict": cnn_best_state,
        "best_epoch": cnn_best_epoch,
        "validation_accuracy_percent": cnn_val_acc,
        "test_accuracy_percent": cnn_test_acc,
        "evaluation_mode": "standard",
        "image_size": IMAGE_SIZE,
    },
    "best_resnet18_transfer_learning.pth": {
        "architecture": "torchvision_resnet18",
        "model_state_dict": resnet_best_state,
        "best_epoch": resnet_best_epoch,
        "validation_accuracy_percent": val_acc,
        "test_accuracy_percent": resnet_test_acc,
        "evaluation_mode": resnet_evaluation_mode,
        "image_size": IMAGE_SIZE,
    },
    "best_efficientnet_b0_vgaf.pth": {
        "architecture": "hsemotion_enet_b0_8_best_vgaf",
        "model_state_dict": efficientnet_best_state,
        "best_epoch": efficientnet_best_epoch,
        "validation_accuracy_percent": efficientnet_val_acc,
        "test_accuracy_percent": efficientnet_test_acc,
        "evaluation_mode": efficientnet_evaluation_mode,
        "image_size": 224,
    },
}
checkpoint_evaluation_keys = {
    "best_cnn_baseline.pth": "cnn_baseline",
    "best_resnet18_transfer_learning.pth": "resnet18_transfer_learning",
    "best_efficientnet_b0_vgaf.pth": "efficientnet_b0_vgaf",
}
saved_model_paths = []
for filename, model_info in checkpoint_specs.items():
    model_report = classification_reports[checkpoint_evaluation_keys[filename]]
    checkpoint = {
        **model_info,
        "best_epoch": int(model_info["best_epoch"]),
        "validation_accuracy_percent": float(model_info["validation_accuracy_percent"]),
        "test_accuracy_percent": float(model_info["test_accuracy_percent"]),
        "class_names": CLASS_NAMES,
        "normalization_mean": imagenet_mean,
        "normalization_std": imagenet_std,
        "classification_metrics": {
            "precision_macro": float(model_report["macro avg"]["precision"]),
            "recall_macro": float(model_report["macro avg"]["recall"]),
            "f1_macro": float(model_report["macro avg"]["f1-score"]),
            "precision_weighted": float(model_report["weighted avg"]["precision"]),
            "recall_weighted": float(model_report["weighted avg"]["recall"]),
            "f1_weighted": float(model_report["weighted avg"]["f1-score"]),
        },
    }
    model_path = RESULTS_DIR / filename
    torch.save(checkpoint, model_path)
    saved_model_paths.append(model_path)
    print(f"Saved: {model_path.name}")

test_paths = [path for path, _ in test_dataset.samples]
relative_test_paths = [str(Path(path).relative_to(PROJECT_ROOT)) for path in test_paths]
for model_key, result in model_evaluations.items():
    report_dict = classification_reports[model_key]
    report_text = classification_report(
        result["y_true"], result["y_pred"], target_names=CLASS_NAMES,
        digits=4, zero_division=0,
    )
    pd.DataFrame(report_dict).transpose().to_csv(
        RESULTS_DIR / f"classification_report_{model_key}.csv", encoding="utf-8-sig"
    )
    (RESULTS_DIR / f"classification_report_{model_key}.txt").write_text(
        report_text, encoding="utf-8"
    )
    predictions_df = pd.DataFrame({
        "image_path": relative_test_paths,
        "true_index": result["y_true"],
        "true_class": [CLASS_NAMES[i] for i in result["y_true"]],
        "predicted_index": result["y_pred"],
        "predicted_class": [CLASS_NAMES[i] for i in result["y_pred"]],
        "correct": result["y_true"] == result["y_pred"],
    })
    predictions_df.to_csv(
        RESULTS_DIR / f"test_predictions_{model_key}.csv",
        index=False, encoding="utf-8-sig",
    )

selected_report = classification_reports[selected_model_name]
pd.DataFrame(selected_report).transpose().to_csv(
    RESULTS_DIR / "classification_report.csv", encoding="utf-8-sig"
)
print("Đã lưu dữ liệu số tại:", RESULTS_DIR)

In [ ]:
# Vẽ độ chính xác và giá trị mất mát theo từng epoch.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for model_name, group in history_df.groupby("model", sort=False):
    axes[0].plot(group["epoch"], group["train_acc"], marker="o", ms=3, label=f"{model_name} train")
    axes[0].plot(group["epoch"], group["val_acc"], marker="o", ms=3, linestyle="--", label=f"{model_name} val")
    axes[1].plot(group["epoch"], group["train_loss"], marker="o", ms=3, label=f"{model_name} train")
    axes[1].plot(group["epoch"], group["val_loss"], marker="o", ms=3, linestyle="--", label=f"{model_name} val")

axes[0].set(title="Accuracy theo epoch", xlabel="Epoch", ylabel="Accuracy (%)")
axes[1].set(title="Loss theo epoch", xlabel="Epoch", ylabel="Cross-entropy loss")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "accuracy_loss_curves.png", dpi=180, bbox_inches="tight")
plt.show()

# Lưu riêng từng biểu đồ để thuận tiện chèn vào báo cáo đồ án.
for metric, ylabel, filename in [
    ("acc", "Accuracy (%)", "accuracy_curve.png"),
    ("loss", "Cross-entropy loss", "loss_curve.png"),
]:
    plt.figure(figsize=(9, 5))
    for model_name, group in history_df.groupby("model", sort=False):
        plt.plot(group["epoch"], group[f"train_{metric}"], label=f"{model_name} train")
        plt.plot(group["epoch"], group[f"val_{metric}"], linestyle="--", label=f"{model_name} val")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.grid(alpha=0.25)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()

In [ ]:
# Lưu ma trận nhầm lẫn theo số lượng và theo tỉ lệ của từng lớp thực tế.
fig, axes = plt.subplots(len(model_evaluations), 2, figsize=(18, 7 * len(model_evaluations)))
for row_index, (model_key, result) in enumerate(model_evaluations.items()):
    cm = confusion_matrix(
        result["y_true"], result["y_pred"], labels=np.arange(NUM_CLASSES)
    )
    cm_normalized = confusion_matrix(
        result["y_true"], result["y_pred"],
        labels=np.arange(NUM_CLASSES), normalize="true",
    )
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        RESULTS_DIR / f"confusion_matrix_{model_key}.csv", encoding="utf-8-sig"
    )
    pd.DataFrame(cm_normalized, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        RESULTS_DIR / f"confusion_matrix_{model_key}_normalized.csv",
        encoding="utf-8-sig",
    )
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[row_index, 0],
    )
    axes[row_index, 0].set(
        title=f"{result['display_name']} - counts - Acc {result['test_acc']:.2f}%",
        xlabel="Predicted", ylabel="Actual",
    )
    sns.heatmap(
        cm_normalized, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[row_index, 1],
    )
    axes[row_index, 1].set(
        title=f"{result['display_name']} - normalized by actual class",
        xlabel="Predicted", ylabel="Actual",
    )

fig.tight_layout()
fig.savefig(RESULTS_DIR / "confusion_matrices_all_models.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
summary = f'''# Kết quả CNN, ResNet18 và EfficientNet-B0

- CNN baseline — Val: {cnn_val_acc:.2f}%, Test: {cnn_test_acc:.2f}%
- ResNet18 — Val: {val_acc:.2f}%, Test: {resnet_test_acc:.2f}%
- EfficientNet-B0 VGAF — Val: {efficientnet_val_acc:.2f}%, Test: {efficientnet_test_acc:.2f}%
- Mô hình được chọn: {selected_model_name}
- Kết quả được chọn: Val {selected_val_acc:.2f}%, Test {selected_test_acc:.2f}% ({selected_evaluation_mode})
- Epoch tốt nhất: {selected_best_epoch}
- Kích thước dữ liệu: Train {len(train_dataset)}, Val {len(val_dataset)}, Test {len(test_dataset)}

Mô hình được chọn theo độ chính xác trên tập xác thực; tập kiểm tra chỉ dùng để đánh giá kết quả cuối cùng.
'''
(RESULTS_DIR / "README.md").write_text(summary, encoding="utf-8")

RUN_FINISHED_AT = datetime.now().astimezone()
run_info.update({
    "status": "completed",
    "finished_at": RUN_FINISHED_AT.isoformat(),
    "duration_seconds": (RUN_FINISHED_AT - RUN_STARTED_AT).total_seconds(),
    "selected_model": selected_model_name,
    "validation_accuracy_percent": float(selected_val_acc),
    "test_accuracy_percent": float(selected_test_acc),
})
(RESULTS_DIR / "run_info.json").write_text(
    json.dumps(run_info, ensure_ascii=False, indent=2), encoding="utf-8"
)

# Chép kết quả mới nhất về thư mục chính để các phần khác của đồ án dùng đường dẫn cũ.
for artifact in RESULTS_DIR.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, RESULTS_ROOT / artifact.name)

print(summary)
print("Thư mục lưu riêng của lần chạy:", RESULTS_DIR)
print("Kết quả mới nhất cũng được cập nhật tại:", RESULTS_ROOT)
print("\nDANH SÁCH ARTIFACT:")
for artifact in sorted(RESULTS_DIR.iterdir()):
    print(f"- {artifact.name}: {artifact.stat().st_size:,} bytes")